# Imports

In [2]:
%load_ext autoreload
%autoreload 2
import sys

sys.path.append('../../')

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import pyaldata as pyal
import tools.dsp as dsp
from tools.params import Params
import tools.dimensionality as dim
import tools.dataTools as dt
from tools.params import colors
import tools.kinematics as kin

from tqdm import tqdm



In [10]:
SESSIONS = [
    # "M103_2026_02_18_15_30",
    # "M103_2026_02_20_16_00",
    "M106_2026_02_25_15_00",
    # "M106_2026_02_27_16_00",
    # 'M086_2025_12_11_15_00',
    # 'M086_2025_12_10_15_00'
]

# all_sess_proc = {}
for sess in SESSIONS:
    td, _ = dsp.load_and_process_session(sess, std=0.03)
    # perturb_td = pyal.restrict_to_interval(
    #     td, start_point_name='idx_sol_on', rel_start=-200, rel_end=300
    # )
    # perturb_td = perturb_td.sample(frac=1, random_state=42).reset_index(drop=False)
    # perturb_td    = dsp.drop_trials_sem_crosses_zero(perturb_td, std=True)
    # all_sess_proc[sess] = perturb_td


/home/jovyan/datascience/workspaces/mesparza-40inbrain-2dneuroelectronics-2ecom/pyaldata/pyaldata/data_cleaning.py:120: UserWarning: fields: ['values_before_camera_trigger', 'idx_before_camera_trigger', 'values_Sol_duration', 'values_Sol_direction'] could not be converted to int.
  utils.warnings.warn(f"fields: {bad_fields} could not be converted to int.")
/home/jovyan/datascience/workspaces/mesparza-40inbrain-2dneuroelectronics-2ecom/pyaldata/pyaldata/data_cleaning.py:120: UserWarning: fields: ['values_before_camera_trigger', 'values_Sol_duration', 'idx_Sol_duration', 'values_Sol_direction', 'idx_Sol_direction', 'idx_sol_on'] could not be converted to int.
  utils.warnings.warn(f"fields: {bad_fields} could not be converted to int.")
/home/jovyan/datascience/workspaces/mesparza-40inbrain-2dneuroelectronics-2ecom/pyaldata/pyaldata/data_cleaning.py:120: UserWarning: fields: ['values_before_camera_trigger', 'values_Sol_duration', 'idx_Sol_duration', 'values_Sol_direction', 'idx_Sol_direct

['CP_spikes', 'MOp_spikes', 'all_spikes', 'VAL_spikes', 'SSp_spikes']
Resulting CP_spikes ephys data shape is (NxT): (206, 48000)
Resulting MOp_spikes ephys data shape is (NxT): (156, 48000)
Resulting all_spikes ephys data shape is (NxT): (11, 48000)
Resulting VAL_spikes ephys data shape is (NxT): (132, 48000)
Resulting SSp_spikes ephys data shape is (NxT): (142, 48000)
add_concat_perturb_time: dropping 1 trial(s) with missing idx_sol_on
Skipped 87 trials
Otsu immobility threshold: 2.2123
Dropped 130 of 499 rows (26.05%).


In [11]:
td.head()

,animal,session,trial_id,trial_name,trial_length,bin_size,idx_trial_start,idx_trial_end,idx_CPI,values_before_camera_trigger,...,SSp_rates_pca,bhv,bhv_concat,concat_perturb_time,concat_trial_start,power,phases,disturb_score,disturb_mean,disturb_sum
1,M106,M106_2026_02_25_15_00,4,trial,600,0.01,49000,49599,[],[],...,"[[92.62157653584134, -5.365383543834213, 64.91...","[[4.293750991903091, 13.685706393891431, 194.8...","[[190.55076207954733, 176.0942658871152, 185.3...",300,100,"[[1.772731172998118, 0.7044306660633013, 2.235...","[[2.6971999348495284, 1.5873911506776348, 2.24...","[-0.8209118566282974, 0.06634696711937461, -0....",-0.548872,0.818652
2,M106,M106_2026_02_25_15_00,6,trial,600,0.01,50100,50699,[],[],...,"[[-27.80002210442575, 10.418235123732273, 42.6...","[[4.848082574702561, 13.849731133630389, 191.6...","[[201.71597906300204, 176.41945003109635, 200....",700,500,"[[1.4390329582106023, 0.8293205909225932, 1.93...","[[1.4955204731201683, -1.4137292780338586, 1.4...","[-0.2546640326883688, 0.08284335570641839, -0....",-0.129664,0.192000
3,M106,M106_2026_02_25_15_00,8,trial,600,0.01,51000,51599,[],[],...,"[[73.91643943060481, 1.1494926031425265, 48.54...","[[4.556679576659612, 12.684983229778044, 198.6...","[[202.35127647582365, 176.16686684428893, 204....",500,300,"[[1.359528028094149, 1.0745439146059905, 1.880...","[[1.6273823102228406, -1.4749047614356465, 1.6...","[-0.03748463238856231, -0.08918353914742244, -...",-0.016381,-0.706474
4,M106,M106_2026_02_25_15_00,10,trial,600,0.01,52100,52699,[],[],...,"[[1.914436712391435, -8.361634777934082, 48.59...","[[15.627078107989789, 12.159316468286775, 194....","[[198.96013990787125, 171.74232332449802, 200....",700,500,"[[1.612407629436079, 0.5659264315893815, 2.081...","[[-1.1328979324063897, -1.6266605312510298, -1...","[-0.3864010343485246, 0.035008640562434454, -0...",-0.358886,0.634138
6,M106,M106_2026_02_25_15_00,14,trial,600,0.01,54100,54699,[],[],...,"[[71.5662861389406, -50.27500514326536, -23.15...","[[12.733264057451535, 13.330876925968552, 201....","[[187.3947220920164, 172.74222693868725, 181.4...",700,500,"[[1.8023051191338444, 0.24364045049745295, 2.0...","[[-1.6633593547812788, 2.4130125548638874, -1....","[-0.7155123721342386, -0.047214784402068714, -...",-0.334870,0.604057
